# Lectures 18-22: Portfolio Workflow in Colab

This notebook combines the portfolio lessons into one Colab-ready workflow:

- Lecture 18: Build the first portfolio in Colab
- Lecture 19: Update the portfolio and review version history
- Lecture 20: Localize the portfolio
- Lecture 21: Final review of the human view and agent-ready YAML
- Lecture 22: Wrap-up, portfolio workflow recap, and next steps

Run the cells from top to bottom. The notebook creates a single workspace folder, builds a portfolio from source lanes, refreshes it after a source change, creates localized HTML pages, validates the machine-readable YAML, and closes with the course wrap-up.

### What We Just Covered

The earlier commands taught the individual parts: validation, generation, fragments, catalogs, and graphs. Lectures 16 and 17 connect those parts to the portfolio builder: a repeatable workflow that takes source material and produces human-reviewable HTML plus machine-readable YAML.

Next, we run that full portfolio workflow in Colab.

# Lecture 18: Build the First Portfolio in Colab

This is the main hands-on portfolio lesson. It loads source material, runs the portfolio builder, inspects generated folders, opens HTML output, and reviews catalog and graph files.

### What We Just Covered

The portfolio builder turns source lanes into a connected portfolio workspace. The source lanes represent objectives, use cases, signals, and product material; the outputs include generated YAML, ODPC catalog content, ODPG graph content, and an HTML review page.

Next, we create the Colab workspace, add source files, and run the first portfolio build.

## Prepare Colab

Install the SDK in the Colab runtime. Re-run this cell if Colab restarts the session.

In [ ]:
!python -m pip install --upgrade open-data-products

## Store The Provider Key

If you use Claude, store `ANTHROPIC_API_KEY` in Colab secrets, then read it into the notebook environment.

Do not commit or share notebooks that contain real API keys.

### Before You Run This

The portfolio builder uses an LLM provider for generation and localization. Keep the API key in Colab secrets, not inside the notebook text. If the key is missing, provider-backed cells will fail or need to be skipped.

In [ ]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Create A Workspace Folder

Keep the portfolio section in one working folder. In Colab, this keeps source files, generated YAML, HTML, localized pages, and version history together instead of scattering files through downloads.

The rest of Lectures 18-22 assume your current folder is this workspace.

### Before You Run This

The portfolio workflow creates many related files: source documents, generated YAML, HTML pages, localized pages, and version snapshots. Keeping them in one Colab workspace folder makes the later review and zip steps much easier.

In [ ]:
!mkdir -p /content/odp-portfolio-workspace
%cd /content/odp-portfolio-workspace

## Create Source Lanes

Create source folders in the notebook runtime. These four lanes match the input map from Lecture 17: objectives, use cases, signals, and products.

### Before You Run This

Source lanes are simple folders that tell the builder what kind of material it is reading. Objectives, use cases, signals, and product notes each play a different role in the final portfolio.

In [ ]:
!mkdir -p source_docs/objectives source_docs/use-cases source_docs/signals source_docs/products

## Add Source Files

Run this shell cell inside `/content/odp-portfolio-workspace` to add one source file to each lane.

In [ ]:
%%bash
cat > source_docs/objectives/reduce-churn-objective.md <<'MD'
# Reduce Preventable Churn

Customer success leaders want to reduce preventable churn by identifying
accounts with declining product usage, unresolved support friction, and renewal
risk before the next business review.
MD

cat > source_docs/use-cases/retention-risk-workflow.md <<'MD'
# Retention Risk Workflow

Customer success managers need a weekly workflow that ranks accounts by churn
risk, explains the main risk drivers, and suggests which accounts should be
contacted before renewal.
MD

cat > source_docs/signals/churn-risk-signal.txt <<'TXT'
Daily customer health note from April 18, 2026 at 09:30.

Product usage is down for several priority accounts, unresolved support tickets
are increasing, and renewal conversations have slowed. The signal should help
retention teams detect preventable churn earlier.
TXT

cat > source_docs/products/customer-health-product.md <<'MD'
# Customer Health Signals Product

The product combines customer profile, subscription status, product usage,
support ticket volume, renewal date, campaign engagement, and churn-risk
signals. It is used by customer success and lifecycle marketing teams for
retention planning.
MD

## Check The Source Files

Use this quick check before building so learners can see what the notebook created.

In [ ]:
!find source_docs -maxdepth 3 -type f | sort

## Run Portfolio Build

This creates a portfolio workspace with ODPC, ODPS, ODPG, HTML, and report artifacts.

### Before You Run This

This is the main orchestration command. Instead of manually generating fragments, building a catalog, building a graph, and rendering HTML one step at a time, `portfolio build` combines those actions into one workflow.

In [ ]:
!open-data-products portfolio build \
  --objectives source_docs/objectives/ \
  --use-cases source_docs/use-cases/ \
  --signals source_docs/signals/ \
  --products source_docs/products/ \
  --title "Customer Intelligence Portfolio" \
  --output portfolio/ \
  --provider claude \
  --model claude-sonnet-4-5

## Inspect Generated Folders

Look for `portfolio/index.html`, `portfolio/portfolio.yaml`, `portfolio/odpc/catalog.yaml`, `portfolio/odpc/fragments/`, `portfolio/odpg/graph.yaml`, and `portfolio/odps/products/`.

### Before You Run This

Do not worry if there are many files. The most important pattern is: HTML is for human review, YAML is for standards-based automation, and version/report files help explain what happened during the build.

In [ ]:
!find portfolio -maxdepth 3 -type f | sort

## Open The HTML Output

Open `portfolio/index.html` from the workspace. In Colab, use the file browser on the left side and open:

```text
/content/odp-portfolio-workspace/portfolio/index.html
```

Avoid downloading only `index.html` by itself. Later lessons add localized pages and translation files beside it, so the whole `portfolio/` folder should stay together. If you need to move the result to your computer, zip the portfolio folder.

In [ ]:
!zip -r portfolio-review.zip portfolio

## Review Catalog And Graph Files

These commands explain the workspace and validate the generated catalog and graph artifacts.

In [ ]:
!open-data-products portfolio explain portfolio/
!open-data-products validate portfolio/odpc/catalog.yaml
!open-data-products validate portfolio/odpg/graph.yaml

## What You Learned

- Portfolio build combines source lanes into one workspace.
- The workspace includes ODPC, ODPS, ODPG, HTML, and report artifacts.
- The browser view is for review, while YAML files remain agent-ready.

## Next Lesson

Continue to Lecture 19: Update the portfolio and review version history.

# Lecture 19: Update the Portfolio and Review Version History

A portfolio is a living artifact. Source material changes, new signals appear, and product drafts improve over time. This lesson shows how to refresh a workspace and review previous versions.

Continue from the same workspace folder you created in Lecture 18:

```text
/content/odp-portfolio-workspace
```

All commands in this lesson assume the current folder contains both `source_docs/` and `portfolio/`.

### What We Just Covered

A portfolio should not be treated as a one-time export. Business priorities change, new signals appear, and reviewers need to compare current and previous outputs. Version history makes those changes easier to review and govern.

Next, we add a new signal, refresh the portfolio, and inspect versioned outputs.

## Add Or Change Source Material

Add a new file to one saved source lane. In Colab, run this in a shell cell from `/content/odp-portfolio-workspace`.

In [ ]:
%%bash
cat > source_docs/signals/support-pressure-signal.txt <<'TXT'
Support operations note from April 19, 2026 at 10:15.

Priority accounts with unresolved tickets are waiting longer for first response.
Customer success teams want this signal linked to retention review workflows.
TXT

## Refresh Changed Sources

Refresh scans saved source lanes and sends changed or new source files to the LLM. Existing unchanged artifacts are preserved where possible.

### Before You Run This

Refresh is for normal portfolio updates. It looks at the saved source lanes and processes changed or new source files while preserving unchanged artifacts where possible.

In [ ]:
!open-data-products portfolio refresh portfolio/ \
  --provider claude \
  --model claude-sonnet-4-5

## Force Full Reprocessing

Use `--all-sources` when the full evidence set should be reprocessed.

### Before You Run This

A full reprocess is heavier than a normal refresh. Use it when you want the complete source set reconsidered, such as after changing model choice, prompts, or the overall interpretation of the portfolio.

In [ ]:
!open-data-products portfolio refresh portfolio/ \
  --all-sources \
  --provider claude \
  --model claude-sonnet-4-5

## Sync Edited YAML Without An LLM

Use sync after directly editing ODPC fragments, ODPS products, or graph YAML.

### Before You Run This

Sync is the non-LLM path. Use it when you manually edit YAML and only need the browser view or derived files rebuilt from those existing artifacts.

In [ ]:
!open-data-products portfolio sync portfolio/

## Review Version History

Successful builds and refreshes snapshot previous portfolio outputs. The latest `index.html` includes links to available versions so reviewers can compare current and previous portfolio pages.

In [ ]:
!find portfolio/versions -maxdepth 2 -type f | sort

## What You Learned

- Refresh processes changed and new source documents by default.
- `--all-sources` forces full reprocessing.
- `portfolio sync` rebuilds browser output from edited YAML without an LLM.
- Version snapshots support review and governance over time.

## Next Lesson

Continue to Lecture 20: How to localize a portfolio.

# Lecture 20: How to Localize a Portfolio

Portfolio localization creates translated static HTML review pages for regional stakeholders. It does not change the canonical ODPC, ODPS, or ODPG YAML files.

Use this lesson after you have a working portfolio workspace from the previous lessons.

Continue from the same workspace folder you created in Lecture 18:

```text
/content/odp-portfolio-workspace
```

Localization writes files into `portfolio/` inside that workspace. Do not download individual HTML files before running localization; keep `index.html`, localized pages, and `portfolio-i18n.yaml` together.

### What We Just Covered

Localization changes the human-facing review experience while keeping the canonical YAML artifacts as the source of truth. It applies to the active portfolio page and does not retroactively translate older `portfolio/versions/` snapshots. If a translated record of a specific version is needed, localize while that version is current and preserve the zipped portfolio package.

Next, we localize the active portfolio page and package the localized review output.

## Start From A Rendered Portfolio

You need a portfolio workspace with a current `index.html`. If the page does not exist yet, render it from the current YAML artifacts.

In [ ]:
!ls portfolio/index.html
!open-data-products portfolio render portfolio/

## Choose Target Languages

`portfolio localize` accepts BCP 47 language tags. Start with one or two languages while learning the workflow:

```text
fi   Finnish
sv   Swedish
ar   Arabic
vi   Vietnamese
```

Arabic is useful for checking right-to-left page rendering. Finnish and Swedish are useful for Nordic stakeholder review examples.

## Run Localization

Use a configured LLM provider to translate the visible HTML strings. The command reads the existing portfolio HTML, translates human-facing text, and writes localized pages beside the main `index.html`.

### Before You Run This

Localization translates the human-facing review page. It should preserve the structure, identifiers, and relationships that make the portfolio reliable for automation and governance.

In [ ]:
!open-data-products portfolio localize portfolio/ \
  --languages "fi,sv" \
  --provider claude \
  --model claude-sonnet-4-5

## Review The Outputs

Check that the localized pages and translation file exist. `portfolio-i18n.yaml` stores the translated strings used to render localized HTML pages. It is generated from the portfolio view, not from editing the canonical YAML artifacts.

In [ ]:
!ls portfolio/index.fi.html
!ls portfolio/index.sv.html
!ls portfolio/portfolio-i18n.yaml

## Open The Localized Pages

Open the localized HTML files from the same workspace:

```text
/content/odp-portfolio-workspace/portfolio/index.fi.html
/content/odp-portfolio-workspace/portfolio/index.sv.html
```

Review the overview, artifact panels, product cards, graph labels, and about section. If you want to move the localized review package to your computer, zip the whole portfolio folder instead of downloading one HTML file at a time.

In [ ]:
!zip -r portfolio-localized-review.zip portfolio

## Keep YAML As The Source Of Truth

Localization is for human-facing review pages. The canonical files remain:

```text
portfolio/portfolio.yaml
portfolio/odpc/catalog.yaml
portfolio/odpc/fragments/*.yaml
portfolio/odpg/graph.yaml
portfolio/odps/products/*.yaml
```

Agents, scripts, validation commands, and version control should keep using those YAML files as the source of truth.

### Before You Run This

Localized HTML helps people review the portfolio in another language, but the canonical YAML files remain the authoritative artifacts for agents, scripts, validation, and version control.

In [ ]:
!find portfolio -path "*/versions/*" -prune -o -name "*.yaml" -print | sort

## Troubleshooting

If localization takes too long with a local model, try fewer languages first. If a command pasted into `zsh` behaves strangely, check that every line continuation backslash is the final character on its line. A space after `\` breaks the command.

In [ ]:
# Optional local-model localization example.
# !open-data-products portfolio localize portfolio/ \
#   --languages "fi" \
#   --provider ollama \
#   --model qwen2.5

## Strict Validation Option

If automation should fail on validation issues, add `--strict-validation`.

In [ ]:
# Optional strict validation example.
# !open-data-products portfolio localize portfolio/ \
#   --languages "fi,sv" \
#   --provider claude \
#   --model claude-sonnet-4-5 \
#   --strict-validation

## What You Learned

- `portfolio localize` creates translated static HTML review pages.
- Localization leaves ODPC, ODPS, and ODPG YAML artifacts unchanged.
- `portfolio-i18n.yaml` stores translated page strings.
- BCP 47 tags such as `fi`, `sv`, `ar`, and `vi` select target languages.
- Local models can work, but hosted providers are often better for longer multilingual review pages.

## Next Lesson

Continue to Lecture 21: Final review: human view and agent-ready YAML.

# Lecture 21: Final Review: Human View and Agent-Ready YAML

The portfolio workflow serves two audiences. The HTML view is for humans. The YAML files are for AI agents, automation, validation, and long-term version control.

Continue from the same workspace folder you created in Lecture 18:

```text
/content/odp-portfolio-workspace
```

### What We Just Covered

The final portfolio package serves two audiences. Humans review the HTML pages, including localized pages when available. Agents, scripts, validators, and version control use the YAML artifacts.

Next, we inspect both sides of the output and run the final validation commands.

## Human Review

Open:

```text
/content/odp-portfolio-workspace/portfolio/index.html
```

Review the overview, artifact types, products, graph, and about tabs. Product cards open detailed views with pricing plans, linked access, SLA, data quality, payment, licensing, and raw artifact references when those sections exist.

If you localized the portfolio in Lecture 20, review those pages from the same workspace too:

```text
/content/odp-portfolio-workspace/portfolio/index.fi.html
/content/odp-portfolio-workspace/portfolio/index.sv.html
```

## Agent-Ready YAML

Review the generated YAML files. These are the key files for agents, automation, validation, and version control.

### Before You Run This

This final review looks beyond the browser page. The YAML files are what another tool, workflow, or AI agent can consume later, so they need to be findable, valid, and consistent.

In [ ]:
!find portfolio -path "*/versions/*" -prune -o -name "*.yaml" -print | sort

The important files are:

- `portfolio/portfolio.yaml`
- `portfolio/odpc/catalog.yaml`
- `portfolio/odpc/fragments/*.yaml`
- `portfolio/odpg/graph.yaml`
- `portfolio/odps/products/*.yaml`

## Validate The Artifacts

Portfolio commands default to warning mode for schema-invalid generated ODPS drafts so users can still review the browser output. Use `--strict-validation` when automation should fail on schema errors.

### Before You Run This

Validation is the final governance check in this workflow. It does not replace human judgment, but it catches structural problems before generated artifacts are reused or shared.

In [ ]:
!open-data-products validate portfolio/odpc/catalog.yaml
!open-data-products validate portfolio/odpg/graph.yaml
!open-data-products portfolio explain portfolio/

## What You Learned

- HTML supports human portfolio review.
- YAML supports AI agents and automation.
- Final review checks both the browser experience and the generated YAML artifacts.
- Catalogs provide structure, graphs provide relationships, and validation supports governance.

## Next Lesson

Continue to Lecture 22: Wrap-up and next steps.

# Lecture 22: Wrap-up and Next Steps

You have completed the course path from SDK basics to a connected, reviewable, agent-ready data product portfolio workflow.

### What We Just Covered

The course path moved from SDK basics to a connected portfolio workflow: setup, validation, vocabulary, LLM configuration, generation, fragments, graphs, catalogs, portfolio build, version history, localization, and final review.

Next, we close by connecting those skills to real portfolio work after the course.

## What You Can Do Now

You can now use the Open Data Products SDK to:

- install and run the CLI
- validate and explain standards files
- use ODPV vocabulary helpers
- configure local and online LLM providers
- generate full ODPS product drafts
- generate ODPC fragments
- build and inspect ODPG graph relationships
- build ODPC catalogs
- create, refresh, sync, localize, render, and review a portfolio workspace
- inspect both browser output and machine-readable YAML artifacts

## How The Pieces Fit

The individual SDK commands give you precise control over one artifact or one step. The portfolio builder combines those capabilities into a repeatable workflow for real portfolio work.

ODPS describes data products. ODPC organizes portfolio catalog objects. ODPG describes relationships. ODPV keeps language consistent.

Together they create a practical pattern: business intent and source material can become human-reviewable HTML and agent-ready YAML.

## Portfolio As A Workflow

You can now treat the portfolio as a workflow, not only as a set of files. That is important because real portfolio work is repeated: source material changes, reviewers ask for updates, regional stakeholders need localized pages, and agents need clean YAML artifacts.

The portfolio command group gives you a repeatable sequence:

```bash
open-data-products portfolio build \
  --objectives source_docs/objectives/ \
  --use-cases source_docs/use-cases/ \
  --signals source_docs/signals/ \
  --products source_docs/products/ \
  --output portfolio/

open-data-products portfolio refresh portfolio/
open-data-products portfolio sync portfolio/
open-data-products portfolio localize portfolio/ \
  --languages "fi,sv" \
  --provider claude \
  --model claude-sonnet-4-5
open-data-products portfolio render portfolio/
open-data-products portfolio explain portfolio/
```

This matters because the SDK keeps the workflow grounded in artifacts:

- source lanes keep business objectives, use cases, signals, and product briefs organized;
- ODPC catalogs describe the portfolio structure;
- ODPS product YAML keeps data product details machine-readable;
- ODPG graphs describe relationships between portfolio objects;
- HTML gives people a reviewable browser view;
- localization creates regional review pages without changing canonical YAML;
- version snapshots support governance and change review over time.

In other words, the portfolio workflow turns scattered source material into a repeatable operating model: build, review, update, localize, validate, explain, and keep improving.

### What We Just Covered

The portfolio workflow is more than a command sequence. It is an operating model: build, review, update, localize, validate, explain, and keep improving as source material and stakeholder needs change.

Next, we summarize how the portfolio command group keeps that workflow grounded in source lanes, catalogs, product YAML, graphs, HTML, localization, and version snapshots.

## Coming Next: ODPR Workflow Recipes

ODPR, the Open Data Product Recipe Specification, is in development now and is published as a draft specification. Because of that, ODPR is not applied as a hands-on standard in this Masterclass. The portfolio workflow you ran in this notebook uses the current SDK portfolio commands directly.

ODPR is the next layer to watch: workflow recipes for repeatable SDK runs. Where the portfolio command group gives you a built-in workflow, ODPR is intended to let teams define their own workflows on top of the SDK.

An ODPR-style recipe could define:

- the ordered SDK steps to run;
- which provider or model to use for each step;
- input and output folders;
- validation gates;
- context formats such as YAML, TOON, or GCF;
- review and localization policy.

For example, a future recipe could capture a release review workflow:

```yaml
recipes:
  release-portfolio-review:
    description: Refresh, localize, render, and explain the release portfolio.
    provider: claude
    steps:
      - command: portfolio.refresh
        workspace: portfolio/
      - command: portfolio.localize
        workspace: portfolio/
        languages: fi,sv
      - command: portfolio.render
        workspace: portfolio/
      - command: portfolio.explain
        workspace: portfolio/
```

The value is simple: instead of copying command sequences between terminals, notebooks, CI jobs, and team documents, a project can name the workflow and run it consistently. ODPR is the direction for making those repeatable workflows explicit, portable, and easier for both humans and AI agents to follow.

Read more from the ODPR draft specification: [Open Data Product Recipe Specification v1.0](https://opendataproducts.org/odpr-v1.0/).

### What We Just Covered

ODPR is the next workflow layer to watch. It is still in draft and is not applied hands-on in this Masterclass, but the idea is important: repeatable SDK workflows can be expressed as named recipes instead of copied manually between notebooks, terminals, CI jobs, and team documents.

Next, we look at a small example of what an ODPR-style recipe could represent.

## Apply This With Your Own Material

Start with source material from a real data product or portfolio idea:

- business objectives
- use cases
- market, operational, customer, quality, or usage signals
- product briefs, emails, transcripts, or governance notes

Put those files into the four source lanes:

```text
odp-portfolio-workspace/source_docs/objectives/
odp-portfolio-workspace/source_docs/use-cases/
odp-portfolio-workspace/source_docs/signals/
odp-portfolio-workspace/source_docs/products/
```

Run `portfolio build` from inside `odp-portfolio-workspace` to create the `portfolio/` workspace output. Review `portfolio/index.html` with people, then inspect the YAML files with agents, scripts, validation commands, or version control.

Then keep the portfolio alive by refreshing sources, syncing edited YAML, and using version snapshots during review. When the audience changes, localize the HTML pages without changing the canonical YAML artifacts.

## Reference Material

Use these references when you want to go deeper:

- [SDK README](../../../README.md)
- [SDK API reference](../../../docs/user/API.md)
- [SDK command guide](../../../docs/user/commands.md)
- [Generation guide](../../../docs/user/generation.md)
- [Portfolio development notes](../../../docs/development/portfolio.md)
- [ODPS product specification](https://opendataproducts.org/v4.1/)
- [ODPC catalog specification](https://opendataproducts.org/odpc-v1.0/)
- [ODPG graph specification](https://opendataproducts.org/odpg-v1.0/)
- [ODPV vocabulary specification](https://opendataproducts.org/odpv-v1.0/)

## Thank You

Thank you for taking the course. If you apply these ideas in your own work, consider sharing your experience, lessons learned, and examples in a blog post or on LinkedIn. Your notes can help other data, analytics, and AI practitioners understand how open data product standards can work in real projects.

You can also connect with me on [LinkedIn](https://ae.linkedin.com/in/jarkkomoilanen) for further discussion.